#  ⭐ `dim_indicacao_meddra`


In [7]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root, fuzzy_merge, hungarian_text_link

import re
import pandas as pd
from unidecode import unidecode

In [8]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [9]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)
bronze["INDICACAO_MEDDRA"] = bronze["INDICACAO_MEDDRA"].fillna("DESCONHECIDO")

bronze = pd.concat(
    [
        (
            bronze[["INDICACAO_MEDDRA"]]
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
        (
            bronze[["INDICACAO_RELATADA_NOTIFICADOR_INICIAL"]].rename(columns={"INDICACAO_RELATADA_NOTIFICADOR_INICIAL": "INDICACAO_MEDDRA"})
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
    ],
    ignore_index=True,
).drop_duplicates()

bronze.to_csv(
    Path(project_root) / "eda/Medicamentos/Medicamentos_INDICACAO_MEDDRA.csv",
    sep=";",
    index=False,
)

In [10]:
bronze.head()

,INDICACAO_MEDDRA,FREQUENCIA_REGISTROS
0,DESCONHECIDO,346290
1,Uso de medicamento para indicação desconhecida,21287
2,Produto usado para indicação desconhecida,18269
3,Doença de Crohn,7328
4,Artrite reumatoide,6544


In [11]:
bronze.INDICACAO_MEDDRA.nunique()

71953

# 🥈 Silver

normalized data, medium quality


In [12]:
dim_soc_llt = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet") 
dim_soc_llt.head()

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [13]:
dim_soc_llt.shape

(18549, 7)

In [14]:
silver = pd.read_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_INDICACAO_MEDDRA.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['INDICACAO_MEDDRA', 'FREQUENCIA_REGISTROS'], dtype='object')

In [15]:
silver.shape

(73804, 2)

In [16]:
hist_silver = fuzzy_merge(
    dim_df = dim_soc_llt[['PK_LLT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT']],
    fact_df = silver,
    dim_id_col="PK_LLT",
    dim_text_col="REACAO_EVTO_ADVERSO_MEDDRA_LLT",
    fact_text_col="INDICACAO_MEDDRA",
    threshold = 98)

KeyError: "None of [Index(['PK_LLT', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT'], dtype='object')] are in the [columns]"

In [ ]:
hist_silver.shape

(69536, 5)

In [ ]:
hist_silver.PK_LLT_MATCH.value_counts(dropna=False)

PK_LLT_MATCH
NaN        60339
16588.0       23
16582.0       20
12629.0       18
7997.0        18
16753.0       18
7746.0        18
16273.0       17
12634.0       17
536.0         16
16292.0       16
16625.0       15
12197.0       14
7400.0        14
5362.0        14
2103.0        14
5693.0        14
17761.0       14
16950.0       14
7415.0        13
17766.0       13
16607.0       13
16833.0       13
5706.0        13
17100.0       12
16760.0       12
1209.0        12
7473.0        12
1575.0        12
10548.0       12
7735.0        12
4368.0        12
11289.0       12
11203.0       11
9704.0        11
12729.0       11
9169.0        11
5704.0        11
17798.0       11
16281.0       11
9635.0        10
1566.0        10
11765.0       10
229.0         10
11292.0       10
17942.0       10
143.0         10
1152.0         9
7742.0         9
7432.0         9
11888.0        9
5713.0         9
5924.0         9
9295.0         9
11519.0        9
5545.0         9
5269.0         9
5932.0         9
1

In [ ]:
hist_silver.head(10)

,INDICACAO_MEDDRA,FREQUENCIA_REGISTROS,INDICACAO_MEDDRA_SCORE,PK_LLT_MATCH,INDICACAO_MEDDRA_MATCH,PK_LLT
1,USO DE MEDICAMENTO PARA INDICAÇÃO DESCONHECIDA,19595,100.0,17870.0,USO DE MEDICAMENTO PARA INDICAÇÃO DESCONHECIDA,17870
2,PRODUTO USADO PARA INDICAÇÃO DESCONHECIDA,16515,100.0,17869.0,PRODUTO USADO PARA INDICAÇÃO DESCONHECIDA,17869
3,DOENÇA DE CROHN,6875,100.0,5362.0,DOENÇA DE CROHN,5362
4,ARTRITE REUMATOIDE,6207,100.0,7746.0,ARTRITE REUMATOIDE,7746
5,HIPERTENSÃO,4399,100.0,11289.0,HIPERTENSÃO,11289
7,ESCLEROSE MÚLTIPLA,4085,100.0,1575.0,ESCLEROSE MÚLTIPLA,1575
8,MIELOMA MÚLTIPLO,3449,100.0,16625.0,MIELOMA MÚLTIPLO,16625
9,ESPONDILITE ANQUILOSANTE,3139,100.0,7806.0,ESPONDILITE ANQUILOSANTE,7806
10,CÂNCER DE MAMA,3119,100.0,16582.0,CÂNCER DE MAMA,16582
11,DOR,2982,100.0,5924.0,DOR,5924


In [ ]:
dim_soc_llt.query("PK_LLT==5362")

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
5361,14,Distúrbios gastrointestinais,Quadros clínicos gastrointestinais inflamatórios,Colite (excl. infecciosa),Doença de Crohn,5362,Doença de Crohn


In [ ]:
dim_soc_llt.query("PK_LLT==7746")

,PK_SOC,SOC,HLGT,HLT,PT,PK_LLT,REACAO_EVTO_ADVERSO_MEDDRA_LLT
7745,17,Distúrbios musculoesqueléticos e do tecido conjuntivo,Distúrbios articulares,Artropatias reumatoides,Artrite reumatoide,7746,Artrite reumatoide


In [ ]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69536 entries, 0 to 69535
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   INDICACAO_MEDDRA        69536 non-null  object 
 1   FREQUENCIA_REGISTROS    69536 non-null  int64  
 2   INDICACAO_MEDDRA_SCORE  69536 non-null  float64
 3   PK_LLT_MATCH            9197 non-null   float64
 4   INDICACAO_MEDDRA_MATCH  9197 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 2.7+ MB


In [ ]:
hist_silver = hist_silver.dropna(subset=["PK_LLT_MATCH"]).copy()
hist_silver["PK_LLT"] = hist_silver["PK_LLT_MATCH"].astype(int)

In [ ]:
hist_silver.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9197 entries, 1 to 69521
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   INDICACAO_MEDDRA        9197 non-null   object 
 1   FREQUENCIA_REGISTROS    9197 non-null   int64  
 2   INDICACAO_MEDDRA_SCORE  9197 non-null   float64
 3   PK_LLT_MATCH            9197 non-null   float64
 4   INDICACAO_MEDDRA_MATCH  9197 non-null   object 
 5   PK_LLT                  9197 non-null   int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 503.0+ KB


In [ ]:
hist_silver.to_parquet(Path(project_root) / "data/02_silver/hist_indicacao_meddra/hist_indicacao_meddra.parquet", index=False)

# 🥇 Gold
